# Final Project : Analyse des actions Tesla & GameStop
### Projet Python pour la Science des Données — IBM Coursera
**Auteur : Ezekiel**

## Importation des bibliothèques

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
print("✅ Bibliothèques importées avec succès !")

## Fonction make_graph (utilisée pour Q5 et Q6)

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1,
                        shared_xaxes=True,
                        subplot_titles=("Prix historique de l'action", "Revenus trimestriels (en millions USD)"),
                        vertical_spacing=0.3)

    stock_data_specific = stock_data[stock_data.Date <= '2021-06-14']
    revenue_data_specific = revenue_data[revenue_data.Date <= '2021-04-30']

    fig.add_trace(go.Scatter(
        x=pd.to_datetime(stock_data_specific.Date),
        y=stock_data_specific.Close.astype("float"),
        name="Prix de clôture",
        line=dict(color='royalblue')
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=pd.to_datetime(revenue_data_specific.Date),
        y=revenue_data_specific.Revenue.astype("float"),
        name="Revenus",
        marker_color='darkorange'
    ), row=2, col=1)

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Prix (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenus (M USD)", row=2, col=1)
    fig.update_layout(
        showlegend=False,
        height=900,
        title=stock,
        xaxis_rangeslider_visible=True
    )
    fig.show()

print("✅ Fonction make_graph définie !")

---
## Question 1 : Extraction des données boursières de Tesla via yfinance

In [ ]:
# Question 1 : Utiliser yfinance pour extraire les données de l'action Tesla
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

---
## Question 2 : Extraction des revenus de Tesla via Webscraping

In [ ]:
# Question 2 : Webscraping des revenus de Tesla
url_tesla = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data_tesla = requests.get(url_tesla).text
soup_tesla = BeautifulSoup(html_data_tesla, "html.parser")

tesla_revenue = pd.DataFrame(columns=["Date", "Revenue"])
rows = []
for row in soup_tesla.find("tbody").find_all("tr"):
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip().replace("$", "").replace(",", "")
        rows.append({"Date": date, "Revenue": revenue})

tesla_revenue = pd.DataFrame(rows)
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue['Revenue'] != ""]
tesla_revenue.tail()

---
## Question 3 : Extraction des données boursières de GameStop via yfinance

In [ ]:
# Question 3 : Utiliser yfinance pour extraire les données de l'action GameStop
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

---
## Question 4 : Extraction des revenus de GameStop via Webscraping

In [ ]:
# Question 4 : Webscraping des revenus de GameStop
url_gme = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.htm"
html_data_gme = requests.get(url_gme).text
soup_gme = BeautifulSoup(html_data_gme, "html.parser")

gme_revenue = pd.DataFrame(columns=["Date", "Revenue"])
rows_gme = []
for row in soup_gme.find("tbody").find_all("tr"):
    cols = row.find_all("td")
    if len(cols) >= 2:
        date = cols[0].text.strip()
        revenue = cols[1].text.strip().replace("$", "").replace(",", "")
        rows_gme.append({"Date": date, "Revenue": revenue})

gme_revenue = pd.DataFrame(rows_gme)
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue['Revenue'] != ""]
gme_revenue.tail()

---
## Question 5 : Dashboard — Actions et Revenus de Tesla

In [ ]:
# Question 5 : Graphique Tesla
make_graph(tesla_data, tesla_revenue, "Tesla (TSLA) — Prix & Revenus")

---
## Question 6 : Dashboard — Actions et Revenus de GameStop

In [ ]:
# Question 6 : Graphique GameStop
make_graph(gme_data, gme_revenue, "GameStop (GME) — Prix & Revenus")

---
## 🔍 Insights Innovants — Au-delà des tâches requises

### Observations clés :

1. **Tesla** affiche une croissance exponentielle régulière, reflétant la confiance des investisseurs dans l'énergie propre et les véhicules électriques.

2. **GameStop** a connu une volatilité extrême en janvier 2021, due au phénomène *short squeeze* amplifié par la communauté Reddit (r/WallStreetBets).

3. **Corrélation revenus/prix** : Tesla montre une forte corrélation entre la croissance de ses revenus et son prix en bourse. GameStop, lui, a vu son prix s'envoler **sans croissance de revenus** — un signal purement spéculatif.

4. **Conclusion** : Ces deux actions illustrent deux réalités opposées du marché financier — croissance fondamentale (Tesla) vs spéculation communautaire (GameStop).

In [ ]:
# BONUS 1 : Rendement cumulé — Tesla vs GameStop
tesla_data['Return'] = tesla_data['Close'].pct_change()
gme_data['Return'] = gme_data['Close'].pct_change()

tesla_data['Cumulative_Return'] = (1 + tesla_data['Return']).cumprod()
gme_data['Cumulative_Return'] = (1 + gme_data['Return']).cumprod()

fig = go.Figure()
fig.add_trace(go.Scatter(x=tesla_data['Date'],
                         y=tesla_data['Cumulative_Return'],
                         name='Tesla', line=dict(color='red')))
fig.add_trace(go.Scatter(x=gme_data['Date'],
                         y=gme_data['Cumulative_Return'],
                         name='GameStop', line=dict(color='blue')))
fig.update_layout(title="Rendement cumulé : Tesla vs GameStop",
                  xaxis_title="Date",
                  yaxis_title="Rendement cumulé")
fig.show()

In [ ]:
# BONUS 2 : Volatilité mobile sur 30 jours
tesla_data['Volatility_30d'] = tesla_data['Close'].rolling(window=30).std()
gme_data['Volatility_30d'] = gme_data['Close'].rolling(window=30).std()

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=tesla_data['Date'],
                          y=tesla_data['Volatility_30d'],
                          name='Tesla - Volatilité', line=dict(color='orange')))
fig2.add_trace(go.Scatter(x=gme_data['Date'],
                          y=gme_data['Volatility_30d'],
                          name='GME - Volatilité', line=dict(color='purple')))
fig2.update_layout(title="Volatilité mobile (30 jours) — Tesla vs GameStop",
                   xaxis_title="Date",
                   yaxis_title="Écart-type du prix")
fig2.show()

In [ ]:
# BONUS 3 : Statistiques descriptives comparatives
print("=" * 50)
print("📊 STATISTIQUES — TESLA (TSLA)")
print("=" * 50)
print(f"Prix moyen    : ${tesla_data['Close'].mean():.2f}")
print(f"Prix max      : ${tesla_data['Close'].max():.2f}")
print(f"Prix min      : ${tesla_data['Close'].min():.2f}")
print(f"Volatilité    : {tesla_data['Close'].std():.2f}")

print("\n" + "=" * 50)
print("📊 STATISTIQUES — GAMESTOP (GME)")
print("=" * 50)
print(f"Prix moyen    : ${gme_data['Close'].mean():.2f}")
print(f"Prix max      : ${gme_data['Close'].max():.2f}")
print(f"Prix min      : ${gme_data['Close'].min():.2f}")
print(f"Volatilité    : {gme_data['Close'].std():.2f}")

---
## Auteur
**Ezekiel** — IBM Data Science Professional Certificate

*Projet réalisé dans le cadre du cours : Python Project for Data Science*